# 03 - KerasTuner and KerasCV Augmentation

This notebook focuses on two practical ideas:

1. **Hyperparameter search** with KerasTuner
2. **Stronger image augmentation pipelines** using KerasCV / modern Keras vision augmentation layers

We keep the search space modest so the notebook remains realistic for Colab.


In [ ]:
# Colab setup
%pip -q install -U tensorflow keras-tuner keras-cv pandas matplotlib


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
import keras_cv

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("TensorFlow:", tf.__version__)
print("keras_tuner:", kt.__version__)
print("keras_cv:", keras_cv.__version__)


## Dataset

We use a smaller CIFAR-10 subset so the tuner can actually finish in Colab without aging into a fossil.


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

x_train = x_train[:18000].astype("float32")
y_train = y_train[:18000]
x_test = x_test[:3000].astype("float32")
y_test = y_test[:3000]

val_size = 3000
x_val, y_val = x_train[-val_size:], y_train[-val_size:]
x_tr, y_tr = x_train[:-val_size], y_train[:-val_size]

print(x_tr.shape, x_val.shape, x_test.shape)


In [ ]:
AUTO = tf.data.AUTOTUNE
batch_size = 128

plain_train_ds = tf.data.Dataset.from_tensor_slices((x_tr, y_tr)).shuffle(5000).batch(batch_size).prefetch(AUTO)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(batch_size).prefetch(AUTO)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size).prefetch(AUTO)


## Visualize KerasCV augmenters


In [ ]:
augmenter = keras.Sequential(
    [
        keras_cv.layers.RandAugment(value_range=(0, 255), num_ops=2, factor=0.5),
        keras_cv.layers.RandomCutout(height_factor=0.2, width_factor=0.2),
    ],
    name="kerascv_augmenter",
)

sample_images = x_tr[:9]
augmented = augmenter(sample_images)

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, image in zip(axes.ravel(), augmented.numpy().astype("uint8")):
    ax.imshow(image)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
def make_augmented_dataset(images, labels, batch_size=128):
    ds = tf.data.Dataset.from_tensor_slices((images, labels)).shuffle(5000).batch(batch_size)

    def apply_aug(x, y):
        x = augmenter(x, training=True)
        x = tf.cast(x, tf.float32) / 255.0
        return x, y

    return ds.map(apply_aug, num_parallel_calls=AUTO).prefetch(AUTO)

def make_plain_dataset(images, labels, batch_size=128, training=False):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        ds = ds.shuffle(5000)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y), num_parallel_calls=AUTO)
    return ds.prefetch(AUTO)

plain_train_ds = make_plain_dataset(x_tr, y_tr, training=True)
aug_train_ds = make_augmented_dataset(x_tr, y_tr)
val_ds = make_plain_dataset(x_val, y_val)
test_ds = make_plain_dataset(x_test, y_test)


## KerasTuner model builder


In [ ]:
def build_tunable_model(hp):
    inputs = keras.Input(shape=(32, 32, 3))
    x = inputs

    for block_idx in range(hp.Int("num_blocks", 2, 3)):
        filters = hp.Choice(f"filters_{block_idx}", [32, 64, 96])
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(
        hp.Choice("dense_units", [128, 256, 384]),
        activation="relu",
    )(x)
    x = layers.Dropout(hp.Float("dropout", 0.2, 0.5, step=0.1))(x)
    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice("learning_rate", [1e-3, 5e-4, 3e-4])
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


In [ ]:
tuner = kt.Hyperband(
    build_tunable_model,
    objective="val_accuracy",
    max_epochs=8,
    factor=3,
    directory="tuner_runs",
    project_name="cifar10_keras_tuner",
    overwrite=True,
)


## Hyperparameter search on the plain dataset


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=2,
    restore_best_weights=True,
)

tuner.search(
    plain_train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[early_stop],
    verbose=1,
)


In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
best_hp.values


In [ ]:
best_plain_model = tuner.hypermodel.build(best_hp)
plain_history = best_plain_model.fit(
    plain_train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop],
    verbose=1,
)
plain_test = best_plain_model.evaluate(test_ds, verbose=0)
plain_test


## Re-train the same best architecture with stronger augmentation

This isolates the effect of augmentation policy from the effect of architecture search.


In [ ]:
best_aug_model = tuner.hypermodel.build(best_hp)
aug_history = best_aug_model.fit(
    aug_train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop],
    verbose=1,
)
aug_test = best_aug_model.evaluate(test_ds, verbose=0)
aug_test


In [ ]:
comparison = pd.DataFrame(
    [
        {"setting": "best_hp + plain_data", "test_loss": plain_test[0], "test_acc": plain_test[1]},
        {"setting": "best_hp + kerascv_aug", "test_loss": aug_test[0], "test_acc": aug_test[1]},
    ]
)
comparison


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(plain_history.history["val_accuracy"], label="plain_val_acc")
plt.plot(aug_history.history["val_accuracy"], label="aug_val_acc")
plt.title("Validation accuracy: plain vs KerasCV augmentation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


## What to discuss in the video

- The tuner searches over filters, dense size, dropout, and learning rate.
- Hyperparameter tuning and augmentation solve different problems:
  - tuning changes the model and optimizer settings
  - augmentation changes the effective training distribution
- KerasCV-style policies such as RandAugment and Cutout often improve robustness by forcing the model to handle nuisance variation.

This notebook is a nice bridge between classical regularization and stronger data-centric training.
